# Notebook 02 - Analisis Exploratorio de Datos (EDA)

**Proyecto:** Dengue en Argentina - EDA (2018-2025)  
**Autora:** Fabiana Grisel Gonzalez  
**Input:** `data/processed/dengue_limpio.csv`  

---

## Preguntas que buscamos responder

1. Cuantos casos de Dengue vs. Zika hay en el dataset?
2. Que provincias concentran mas casos?
3. Como se distribuyen los casos por grupo de edad?
4. Existe un patron estacional (semanas epidemiologicas)?
5. Como evolucionaron los casos anio a anio?

## 1. Configuracion y carga de datos limpios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))
from utils import configurar_graficos, guardar_figura

configurar_graficos()
pd.set_option('display.max_columns', None)

# Cargar dataset limpio
df = pd.read_csv('../data/processed/dengue_limpio.csv')
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

# Variables de referencia
COL_CANTIDAD = 'cantidad_casos'
COL_SEMANA = 'semanas_epidemiologicas'
COL_ANIO = 'ano'

df.head()

In [ ]:
df.info()

## 2. Panorama general del dataset

In [ ]:
print('=' * 60)
print('RESUMEN GENERAL')
print('=' * 60)
print(f'Total de registros:        {len(df):,}')
print(f'Total de casos:            {df[COL_CANTIDAD].sum():,.0f}')
print(f'Rango de anios:            {df[COL_ANIO].min()} - {df[COL_ANIO].max()}')
print(f'Provincias:                {df["provincia_nombre"].nunique()}')
print(f'Departamentos:             {df["departamento_nombre"].nunique()}')
print(f'Eventos:                   {df["evento_nombre"].unique().tolist()}')
print(f'Grupos de edad:            {df["grupo_edad_desc"].nunique()}')
print(f'Semanas epidemiologicas:   {df[COL_SEMANA].min()} - {df[COL_SEMANA].max()}')
print('=' * 60)

## 3. Dengue vs. Zika - Proporcion de eventos

In [ ]:
# Total de casos por tipo de evento
eventos = df.groupby('evento_nombre')[COL_CANTIDAD].sum()
print(eventos)
print(f'\nProporcion Dengue: {eventos.iloc[0] / eventos.sum() * 100:.1f}%')

In [ ]:
# Grafico de torta: Dengue vs Zika
fig, ax = plt.subplots(figsize=(7, 7))
colores = ['#E63946', '#457B9D']
eventos.plot(kind='pie', autopct='%1.1f%%', colors=colores,
             startangle=90, ax=ax, textprops={'fontsize': 13})
ax.set_title('Distribucion de casos: Dengue vs. Zika', fontweight='bold', fontsize=15)
ax.set_ylabel('')
plt.tight_layout()
guardar_figura(fig, 'dengue_vs_zika')
plt.show()

## 4. Distribucion de casos por provincia

In [ ]:
# Casos por provincia
casos_prov = df.groupby('provincia_nombre')[COL_CANTIDAD].sum().sort_values(ascending=False)
print('Casos por provincia:\n')
print(casos_prov)

In [ ]:
# Barplot horizontal - Casos por provincia
fig, ax = plt.subplots(figsize=(10, 7))
casos_prov_sorted = casos_prov.sort_values(ascending=True)
colores = sns.color_palette('YlOrRd', len(casos_prov_sorted))
casos_prov_sorted.plot(kind='barh', ax=ax, color=colores)
ax.set_title('Casos confirmados de Dengue/Zika por provincia', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de casos')
ax.set_ylabel('')

for i, v in enumerate(casos_prov_sorted):
    ax.text(v + casos_prov_sorted.max() * 0.01, i, f'{v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
guardar_figura(fig, 'casos_por_provincia')
plt.show()

## 5. Evolucion temporal - Casos por anio

In [ ]:
casos_anio = df.groupby(COL_ANIO)[COL_CANTIDAD].sum()
print('Evolucion por anio:')
print(casos_anio)

In [ ]:
# Grafico de barras - Evolucion anual
fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#E63946' if v == casos_anio.max() else '#457B9D' for v in casos_anio.values]
casos_anio.plot(kind='bar', ax=ax, color=colores, edgecolor='white', linewidth=0.5)
ax.set_title('Casos confirmados de Dengue/Zika por anio', fontweight='bold', fontsize=14)
ax.set_xlabel('Anio')
ax.set_ylabel('Cantidad de casos')
ax.tick_params(axis='x', rotation=0)

for i, v in enumerate(casos_anio.values):
    ax.text(i, v + casos_anio.max() * 0.02, f'{v:,.0f}', ha='center', fontsize=10, fontweight='bold')

anio_pico = casos_anio.idxmax()
ax.annotate(f'Pico: {anio_pico}', xy=(list(casos_anio.index).index(anio_pico), casos_anio.max()),
            xytext=(0, 30), textcoords='offset points', ha='center',
            fontsize=11, fontweight='bold', color='#E63946',
            arrowprops=dict(arrowstyle='->', color='#E63946'))

plt.tight_layout()
guardar_figura(fig, 'evolucion_anual')
plt.show()

## 6. Estacionalidad - Distribucion por semana epidemiologica

In [ ]:
# Casos por semana epidemiologica (todos los anios sumados)
casos_semana = df.groupby(COL_SEMANA)[COL_CANTIDAD].sum()

fig, ax = plt.subplots(figsize=(14, 5))
casos_semana.plot(kind='bar', ax=ax, color='#E63946', alpha=0.8, edgecolor='white')
ax.set_title('Casos confirmados por semana epidemiologica (todos los anios)', fontweight='bold', fontsize=14)
ax.set_xlabel('Semana epidemiologica')
ax.set_ylabel('Cantidad de casos')
ax.tick_params(axis='x', rotation=0, labelsize=8)

ax.axvspan(0, 20, alpha=0.1, color='red', label='Temporada alta (verano-otonio)')
ax.legend()

plt.tight_layout()
guardar_figura(fig, 'estacionalidad_semana_epi')
plt.show()

In [ ]:
# Heatmap: semana epidemiologica x anio
pivot = df.pivot_table(index=COL_SEMANA, columns=COL_ANIO,
                       values=COL_CANTIDAD, aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, linecolor='white',
            annot=True, fmt='.0f', ax=ax, cbar_kws={'label': 'Casos'})
ax.set_title('Mapa de calor: Semana epidemiologica x Anio', fontweight='bold', fontsize=14)
ax.set_xlabel('Anio')
ax.set_ylabel('Semana epidemiologica')
plt.tight_layout()
guardar_figura(fig, 'heatmap_semana_anio')
plt.show()

## 7. Distribucion por grupo de edad

In [ ]:
casos_edad = df.groupby('grupo_edad_desc')[COL_CANTIDAD].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
casos_edad.sort_values().plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(casos_edad)))
ax.set_title('Casos confirmados por grupo de edad', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de casos')
ax.set_ylabel('')

for i, v in enumerate(casos_edad.sort_values()):
    ax.text(v + casos_edad.max() * 0.01, i, f'{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
guardar_figura(fig, 'casos_por_edad')
plt.show()

## 8. Top 10 departamentos mas afectados

In [ ]:
top_deptos = df.groupby(['provincia_nombre', 'departamento_nombre'])[COL_CANTIDAD].sum()
top_deptos = top_deptos.nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
labels = [f'{depto} ({prov})' for prov, depto in top_deptos.index]
ax.barh(labels, top_deptos.values, color=sns.color_palette('OrRd_r', 10))
ax.set_title('Top 10 departamentos con mas casos confirmados', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de casos')

for i, v in enumerate(top_deptos.values):
    ax.text(v + top_deptos.max() * 0.01, i, f'{v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
guardar_figura(fig, 'top10_departamentos')
plt.show()

## 9. Evolucion por provincia (Top 5) - Small Multiples

In [ ]:
top5_prov = df.groupby('provincia_nombre')[COL_CANTIDAD].sum().nlargest(5).index.tolist()
df_top5 = df[df['provincia_nombre'].isin(top5_prov)]

evol_top5 = df_top5.pivot_table(index=COL_ANIO, columns='provincia_nombre',
                                 values=COL_CANTIDAD, aggfunc='sum', fill_value=0)

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
colores = ['#E63946', '#F4A261', '#2A9D8F', '#264653', '#E9C46A']

for i, prov in enumerate(top5_prov):
    axes[i].bar(evol_top5.index, evol_top5[prov], color=colores[i], edgecolor='white')
    axes[i].set_title(prov, fontweight='bold', fontsize=11)
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    if i == 0:
        axes[i].set_ylabel('Casos')

fig.suptitle('Evolucion anual - Top 5 provincias', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
guardar_figura(fig, 'evolucion_top5_provincias')
plt.show()

## 10. Resumen de hallazgos

| # | Hallazgo | Detalle |
|---|----------|---------|
| 1 | (completar) | (completar) |
| 2 | (completar) | (completar) |
| 3 | (completar) | (completar) |
| 4 | (completar) | (completar) |
| 5 | (completar) | (completar) |

---

**Siguiente paso:** `03_visualizaciones.ipynb` - Visualizaciones finales con storytelling.

![Dashboard Resumen](outputs/figures/dashboard_resumen.png)